# Previsão por Regressão Linear Simples

**Capítulo aplicado:** 7 (Regressão linear simples).

Implementa em **Python puro** (sem NumPy, Pandas ou sklearn) o
**Método dos Mínimos Quadrados** com as fórmulas exatas do capítulo:

$$\\beta_1 = \\frac{\\sum (x_i - \\bar{x})(y_i - \\bar{y})}{\\sum (x_i - \\bar{x})^2}$$

$$\\beta_0 = \\bar{y} - \\beta_1 \\bar{x}$$

$$R^2 = 1 - \\frac{\\sum (y_i - \\hat{y}_i)^2}{\\sum (y_i - \\bar{y})^2}$$

## 1. Funções auxiliares (cap. 3 — subalgoritmos)

In [ ]:
def media(lista):
    if not lista:
        return 0.0
    return sum(lista) / len(lista)


def desvios(lista, media_valor):
    return [x - media_valor for x in lista]

## 2. Ajuste pelo Método dos Mínimos Quadrados

In [ ]:
def ajustar_reta(x, y):
    """Calcula (beta0, beta1) da reta y = beta0 + beta1*x."""
    if len(x) != len(y) or len(x) == 0:
        raise ValueError("Listas x e y devem ter o mesmo tamanho (>0).")

    x_med = media(x)
    y_med = media(y)
    dx = desvios(x, x_med)
    dy = desvios(y, y_med)

    numerador = sum(dx[i] * dy[i] for i in range(len(x)))
    denominador = sum(d * d for d in dx)

    if denominador == 0:
        return y_med, 0.0

    beta1 = numerador / denominador
    beta0 = y_med - beta1 * x_med
    return beta0, beta1


def prever(beta0, beta1, x_novo):
    return beta0 + beta1 * x_novo

## 3. Qualidade do modelo: R² e resíduos

In [ ]:
def r_quadrado(x, y, beta0, beta1):
    y_med = media(y)
    sqe = sum((y[i] - (beta0 + beta1 * x[i])) ** 2 for i in range(len(x)))
    sqt = sum((yi - y_med) ** 2 for yi in y)
    if sqt == 0:
        return 1.0
    return 1 - (sqe / sqt)


def residuos(x, y, beta0, beta1):
    return [y[i] - (beta0 + beta1 * x[i]) for i in range(len(x))]

## 4. Função de alto nível: previsão eólica

In [ ]:
def prever_energia_eolica(historico_vento, historico_geracao, vento_previsto):
    b0, b1 = ajustar_reta(historico_vento, historico_geracao)
    estimativa = prever(b0, b1, vento_previsto)
    r2 = r_quadrado(historico_vento, historico_geracao, b0, b1)
    return {
        "vento_previsto": vento_previsto,
        "energia_estimada": round(estimativa, 2),
        "beta0": round(b0, 3),
        "beta1": round(b1, 3),
        "r2": round(r2, 3),
    }

## 5. Demonstração

Reproduz o exemplo do desafio: a partir do histórico de vento e
geração, prever a energia para vento = 11 m/s.

In [ ]:
vento   = [8, 10, 12, 13, 15]
geracao = [18, 21, 25, 27, 31]

resultado = prever_energia_eolica(vento, geracao, vento_previsto=11)
print("Reta ajustada: energia =", resultado["beta1"], "* vento +", resultado["beta0"])
print("Qualidade (R2):", resultado["r2"])
print("Para vento = 11 -> energia estimada ~", resultado["energia_estimada"])

print("\nResiduos (cap. 7):")
b0, b1 = ajustar_reta(vento, geracao)
for v, g, r in zip(vento, geracao, residuos(vento, geracao, b0, b1)):
    print(f"  vento={v}, real={g}, previsto={round(b0+b1*v,2)}, residuo={round(r,3)}")